<a href="https://colab.research.google.com/github/logiha/northstar-database-analytics/blob/main/NorthStar_MongoDBipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.1 MB/s eta 0:00:00


In [2]:
from pymongo import MongoClient
import pandas as pd

In [6]:
import os
os.listdir('/')

['root',
 'etc',
 'sbin',
 'run',
 'media',
 'proc',
 'lib32',
 'srv',
 'sys',
 'bin',
 'dev',
 'home',
 'lib64',
 'var',
 'libx32',
 'usr',
 'mnt',
 'boot',
 'tmp',
 'opt',
 'lib',
 'content',
 'kaggle',
 '.dockerenv',
 'tools',
 'datalab',
 'python-apt',
 'python-apt.tar.xz']

In [9]:
import os
os.listdir('/content')

['.config',
 'deliveries.csv',
 'customers.csv',
 'vehicles.csv',
 'README.txt',
 'app_events.csv',
 'drivers.csv',
 'orders.csv',
 'incidents.csv',
 'hubs.csv',
 'data_dictionary.csv',
 'complaints.csv',
 'sample_data']

In [10]:
import pandas as pd

deliveries = pd.read_csv('/content/deliveries.csv')

deliveries.head()

,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
0,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
1,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
2,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51
3,DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08.103702,Delayed,16.42,0,0,4.18,13.62
4,DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34.131056,OnTime,14.52,1,0,4.18,9.22


In [11]:
delivery_docs = deliveries.to_dict("records")

delivery_docs[0]

{'delivery_id': 'DL00001',
 'order_id': 'O00938',
 'driver_id': 'D004',
 'vehicle_id': 'V056',
 'hub_id': 'H05',
 'dispatch_time': '2024-06-18 10:57:00',
 'delivery_completed_at': '2024-06-19 09:05:59.904311',
 'delivery_status': 'Failed',
 'route_distance_km': 17.26,
 'manual_route_override_count': 1,
 'proof_of_completion_missing': 0,
 'customer_rating_post_delivery': 3.07,
 'fuel_or_charge_cost': 12.05}

In [12]:
len(delivery_docs)

950

In [13]:
!pip install mongomock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00


In [14]:
import mongomock

client = mongomock.MongoClient()
db = client["northstar_db"]

delivery_collection = db["deliveries"]

delivery_collection.insert_many(delivery_docs)

delivery_collection.count_documents({})

950

In [15]:
# Find delayed deliveries

list(
    delivery_collection.find(
        {"delivery_status": "Delayed"}
    ).limit(5)
)

[{'delivery_id': 'DL00004',
  'order_id': 'O00313',
  'driver_id': 'D116',
  'vehicle_id': 'V055',
  'hub_id': 'H02',
  'dispatch_time': '2024-03-08 23:31:00',
  'delivery_completed_at': '2024-03-09 23:30:08.103702',
  'delivery_status': 'Delayed',
  'route_distance_km': 16.42,
  'manual_route_override_count': 0,
  'proof_of_completion_missing': 0,
  'customer_rating_post_delivery': 4.18,
  'fuel_or_charge_cost': 13.62,
  '_id': ObjectId('6a12a72872d2ac021c56bdb4')},
 {'delivery_id': 'DL00006',
  'order_id': 'O00029',
  'driver_id': 'D037',
  'vehicle_id': 'V098',
  'hub_id': 'H03',
  'dispatch_time': '2024-09-11 12:40:00',
  'delivery_completed_at': '2024-09-12 17:11:52.384869',
  'delivery_status': 'Delayed',
  'route_distance_km': 13.84,
  'manual_route_override_count': 0,
  'proof_of_completion_missing': 0,
  'customer_rating_post_delivery': 1.57,
  'fuel_or_charge_cost': 9.58,
  '_id': ObjectId('6a12a72872d2ac021c56bdb6')},
 {'delivery_id': 'DL00007',
  'order_id': 'O00097',
  'dr

In [16]:
# Count failed deliveries

delivery_collection.count_documents(
    {"delivery_status": "Failed"}
)

132

In [17]:
# Aggregation pipeline

pipeline = [
    {
        "$group": {
            "_id": "$delivery_status",
            "average_rating": {
                "$avg": "$customer_rating_post_delivery"
            }
        }
    }
]

list(delivery_collection.aggregate(pipeline))

[{'average_rating': nan, '_id': 'Delayed'},
 {'average_rating': nan, '_id': 'Failed'},
 {'average_rating': nan, '_id': 'OnTime'}]

In [18]:
deliveries['customer_rating_post_delivery'] = pd.to_numeric(
    deliveries['customer_rating_post_delivery'],
    errors='coerce'
)

deliveries_clean = deliveries.dropna(
    subset=['customer_rating_post_delivery']
)

delivery_docs = deliveries_clean.to_dict("records")

delivery_collection.delete_many({})

delivery_collection.insert_many(delivery_docs)

delivery_collection.count_documents({})

936

In [19]:
pipeline = [
    {
        "$group": {
            "_id": "$delivery_status",
            "average_rating": {
                "$avg": "$customer_rating_post_delivery"
            }
        }
    }
]

list(delivery_collection.aggregate(pipeline))

[{'average_rating': 3.11497461928934, '_id': 'Delayed'},
 {'average_rating': 3.0493129770992367, '_id': 'Failed'},
 {'average_rating': 4.28327302631579, '_id': 'OnTime'}]

In [20]:
pipeline = [
    {
        "$group": {
            "_id": "$delivery_status",
            "average_override": {
                "$avg": "$manual_route_override_count"
            }
        }
    }
]

list(delivery_collection.aggregate(pipeline))

[{'average_override': 1.0710659898477157, '_id': 'Delayed'},
 {'average_override': 1.0229007633587786, '_id': 'Failed'},
 {'average_override': 0.9276315789473685, '_id': 'OnTime'}]

In [22]:
delivery_collection.create_index("delivery_status")

'delivery_status_1'

In [24]:
delivery_collection.create_index([
    ("hub_id", 1),
    ("delivery_status", 1)
])

'hub_id_1_delivery_status_1'

In [23]:
delivery_collection.index_information()

{'_id_': {'key': [('_id', 1)], 'v': 2},
 'delivery_status_1': {'key': [('delivery_status', 1)], 'v': 2}}